In [ ]:
# Imports
import pandas as pd
import numpy as np
from pathlib import Path

from src.schema import DATE_COL, STORE_COL, ITEM_COL, SALES_COL
from src.backtest import rolling_origin_windows, evaluate_forecast, run_backtest
from src.metrics import wape, rmse, mape, bias


In [ ]:
# Load and prepare training data
df_raw = pd.read_csv('../data/raw/train.csv')
df_raw[DATE_COL] = pd.to_datetime(df_raw[DATE_COL])

df = df_raw.sort_values(
    [STORE_COL, ITEM_COL, DATE_COL]
).reset_index(drop=True)


In [ ]:
# Generate rolling validation windows
windows = rolling_origin_windows(df, horizon=28, n_windows=3)


In [ ]:
# Load external forecasts and combine forecasts for each rolling window
def load_external_forecast(filepath):
    fc = pd.read_csv(filepath)
    fc[DATE_COL] = pd.to_datetime(fc[DATE_COL])
    return fc[[DATE_COL, STORE_COL, ITEM_COL, 'y_hat', 'model']]

classical_bt = load_external_forecast(
    '../data/processed/forecasts_classical_backtest.csv'
)
baselines_fc = load_external_forecast(
    '../data/processed/forecasts_baselines.csv'
)

top10_keys = classical_bt[[STORE_COL, ITEM_COL]].drop_duplicates()

def combined_forecast(train_df, val_df):
    val_start = val_df[DATE_COL].min()
    val_end = val_df[DATE_COL].max()

    classical_window = classical_bt[
        (classical_bt[DATE_COL] >= val_start) &
        (classical_bt[DATE_COL] <= val_end)
    ].copy()

    baselines_window = baselines_fc[
        (baselines_fc[DATE_COL] >= val_start) &
        (baselines_fc[DATE_COL] <= val_end)
    ].merge(top10_keys, on=[STORE_COL, ITEM_COL])

    return pd.concat(
        [classical_window, baselines_window],
        ignore_index=True
    )

comparison_table, detailed = run_backtest(combined_forecast, df)


In [ ]:
# Save backtesting results
comparison_columns = ['model', 'wape', 'rmse', 'mape', 'bias', 'rank']

comparison_table[comparison_columns].to_csv(
    '../data/processed/model_comparison_shryes.csv',
    index=False
)
